# LLM as a Judge with evidently.ai

> **Bazzite-AI Setup Required**  
> Run `D0_00_Bazzite_AI_Setup.ipynb` first to configure Ollama and verify GPU access.

Attribution & License

This notebook is adapted from: [evidentlyai/community-examples](https://github.com/evidentlyai/community-examples.git), licensed under the Apache License, Version 2.0. © Original authors.

Modifications: by Simeon Harrison/EuroCC Austria, © 2025.

In [70]:
#!pip install evidently[llm]

[No output generated]

In [71]:
import pandas as pd
import numpy as np

from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently import BinaryClassification
from evidently.descriptors import *
from evidently.presets import TextEvals, ValueStats, ClassificationPreset, DataSummaryPreset
from evidently.metrics import *

from evidently.llm.templates import BinaryClassificationPromptTemplate

from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import DashboardPanelPlot

from evidently.ui.workspace import CloudWorkspace

[No output generated]

In [72]:
# === Datasets Path ===
from pathlib import Path
DATASETS_DIR = Path("./datasets")
print(f"Datasets: {[f.name for f in DATASETS_DIR.glob('*.csv')]}")

Datasets: ['booking_queries_dataset.csv', 'code_review_dataset.csv', 'health_and_fitness_qna.csv']


In this tutorial, we will:
- Define the evaluation criteria for our LLM judge
- Build an LLM-as-a-Judge using different prompts/models
- Evaluate the quality of the judge comparing results to human labels

# (Optional) Set up Evidently Cloud

Set up API keys for LLM judges:

In [73]:
## import os
## os.environ["OPENAI_API_KEY"] = "OPEN_AI_API_KEY"
## os.environ["ANTHROPIC_API_KEY"] = "ANTHROPIC_API_KEY"

[No output generated]

**Optional**. Connect to Cloud and create a Project:

In [74]:
# ws = CloudWorkspace(token="YOUR_API_TOKEN", url="https://app.evidently.cloud")

[No output generated]

In [75]:
#project = ws.create_project("My project name", org_id="YOUR_ORG_ID")
#project.description = "My project description"

# or project = ws.get_project("PROJECT_ID")

[No output generated]

# Prepare the dataset

We start with an expert-labeled dataset. We will use it as the ground truth for our LLM judge.

In [76]:
dataset_path = DATASETS_DIR / "code_review_dataset.csv"
review_dataset = pd.read_csv(dataset_path)

[No output generated]

Preview:

In [77]:
pd.set_option('display.max_colwidth', None)
review_dataset.head(10)

                                                                                                                                                                                                                                                                                                    Generated review  \
0                                                                                                                                                                  This implementation appears to work, but the approach used does not align with modern best practices. There are ways to make this more efficient.   
1                                                                                                                                                                                                                                                                                             Great job! Keep it up!   
2                                                               

Create an Evidently dataset:

In [78]:
definition = DataDefinition(
    text_columns=["Generated review", "Expert comment"],
    categorical_columns=["Expert label"]
    )

[No output generated]

In [79]:
eval_dataset = Dataset.from_pandas(
    pd.DataFrame(review_dataset),
    data_definition=definition)

[No output generated]

Preview the distribution of classes:

In [80]:
report = Report([
  ValueStats(column="Expert label")
])

my_eval = report.run(eval_dataset)
my_eval

**Optional**. Let's upload the source dataset to Evidently Cloud.

```
ws.add_dataset(
        dataset = eval_dataset,
        name = "source_dataset",
        project_id = project.id,
        description = "Dataset with expert labels on review quality"
        )
```

# Our goal: create LLM judge to match the human labels

**Options**:
- Splitting criteria: (actionable / non-actionable, appropriate tone / inappropriate tone).
- Try create a good/bad judge. (It may be useful to introduce a borderline or "needs review" tag for subtle or new cases).

# Exp 1. Design the LLM judge - First try

For the tutorial flow, we'll keep the steps explicit and run 5 sequential experiments.

First attempt to create the judge:

In [ ]:
import os
from typing import Dict, Any, List, Optional
from evidently.llm.utils.wrapper import OpenAIOptions, OpenAIWrapper, LLMMessage, LLMResult

# === Ollama Configuration via OpenAI-Compatible API ===
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://ov-ollama:11434")
os.environ["OLLAMA_HOST"] = OLLAMA_HOST

# === Model Configuration ===
HF_LLM_MODEL = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO-GGUF"
OLLAMA_LLM_MODEL = f"hf.co/{HF_LLM_MODEL}:Q4_K_M"

OLLAMA_OPTIONS = OpenAIOptions(
    api_key="ollama",
    api_url=f"{OLLAMA_HOST}/v1"
)

# === Patch OpenAIWrapper for smart JSON mode detection ===
# Evidently's OpenAI wrapper doesn't enable JSON mode by default.
# This patch detects when JSON output is expected and enables it.
_original_openai_complete = OpenAIWrapper.complete

async def _json_aware_complete(self, messages: List[LLMMessage], seed: Optional[int] = None) -> LLMResult[str]:
    import openai
    from openai.types.chat.chat_completion import ChatCompletion
    
    message_text = " ".join(m.content for m in messages if m.content)
    needs_json = "json" in message_text.lower() or '"category"' in message_text
    needs_xml = "<new_prompt>" in message_text
    
    formatted_messages = [{"role": msg.role, "content": msg.content} for msg in messages]
    
    try:
        kwargs = {"model": self.model, "messages": formatted_messages, "seed": seed}
        if needs_json and not needs_xml:
            kwargs["response_format"] = {"type": "json_object"}
        
        response: ChatCompletion = await self.client.chat.completions.create(**kwargs)
    except openai.RateLimitError as e:
        from evidently.llm.utils.wrapper import LLMRateLimitError
        raise LLMRateLimitError(e.message) from e
    except openai.APIError as e:
        from evidently.llm.utils.wrapper import LLMRequestError
        raise LLMRequestError(f"Failed to call OpenAI complete API: {e.message}", original_error=e) from e

    content = response.choices[0].message.content
    assert content is not None
    if response.usage is None:
        return LLMResult(content, 0, 0)
    return LLMResult(content, response.usage.prompt_tokens, response.usage.completion_tokens)

OpenAIWrapper.complete = _json_aware_complete

print(f"Ollama host: {OLLAMA_HOST}")
print(f"Model: {OLLAMA_LLM_MODEL}")
print(f"Using OpenAI-compatible API with smart JSON mode detection")

In [82]:
# 1. Name the experiment
name = "naive_prompt"

# 2. Define LLM judge prompt template
feedback_quality = BinaryClassificationPromptTemplate(
        pre_messages=[("system", "You are evaluating the quality of code reviews given to junior developers.")],
        criteria = """An review is GOOD when it's actionable and constructive.
        A review is BAD when is non-actionable or overly critical.
        """,
        target_category="bad",
        non_target_category="good",
        uncertainty="unknown",
        include_reasoning=True,
        )

# 3. Apply the LLM judge
eval_dataset = Dataset.from_pandas(
    pd.DataFrame(review_dataset),
    data_definition=definition,
    descriptors=[
        LLMEval("Generated review",
                template=feedback_quality,
                provider="openai",  # Use OpenAI provider with Ollama's OpenAI-compatible API
                model=OLLAMA_LLM_MODEL,
                alias="LLM-judged quality")
    ],
    options=OLLAMA_OPTIONS
)

# 4. Add TRUE/FALSE for judge alignment
eval_dataset.add_descriptors([
    ExactMatch(columns=["LLM-judged quality", "Expert label"], alias="Judge_alignment")
])

[No output generated]

In [83]:
#print(feedback_quality.get_template())

[No output generated]

**LLM judgments**. View all results locally:

In [84]:
eval_dataset.as_dataframe()

                                                                                                                                                                                                                                                                                                     Generated review  \
0                                                                                                                                                                   This implementation appears to work, but the approach used does not align with modern best practices. There are ways to make this more efficient.   
1                                                                                                                                                                                                                                                                                              Great job! Keep it up!   
2                                                            

**Report**. Let's summarize:

In [85]:
report = Report([
    TextEvals()
])

my_eval = report.run(eval_dataset)
my_eval

**Classification quality**. This function runs the Classification Report to evaluate the LLM judge quality and optionally uploads it to Evidently Cloud (if the workspace is set) with a tag.


In [86]:
def run_classification_report(eval_dataset, name=None, cloud_ws=None, project_id=None):

    df = eval_dataset.as_dataframe()

    # Filter out UNKNOWN predictions
    df_filtered = df[df["LLM-judged quality"] != "UNKNOWN"].copy()

    # Normalize case: LLM outputs uppercase (GOOD/BAD), Expert labels are lowercase (good/bad)
    df_filtered["LLM-judged quality"] = df_filtered["LLM-judged quality"].str.lower()

    # Set the classification Data Definition
    definition_class = DataDefinition(
        classification=[BinaryClassification(
            target="Expert label",
            prediction_labels="LLM-judged quality",
            pos_label="bad"
        )],
        categorical_columns=["Expert label", "LLM-judged quality"]
    )

    # Create a Dataset object
    eval_data = Dataset.from_pandas(df_filtered, data_definition=definition_class)

    # Build classification report
    report = Report([
        ClassificationPreset(),
        ValueStats("LLM-judged quality"),
        ValueStats("Expert label")
    ])

    # Apply tag(s)
    tags = [name] if name else []

    my_eval = report.run(eval_data, tags=tags)

    # Optional: upload to Evidently Cloud
    if cloud_ws and project_id:
        cloud_ws.add_run(project_id, my_eval, include_data=True)

    return my_eval

[No output generated]

(See all Evidently Metrics and Presets: https://docs.evidentlyai.com/metrics/all_metrics)

Run the function to evaluate the LLM judge quality:

In [87]:
my_eval = run_classification_report(
    eval_dataset,
    name=name,
    #cloud_ws=ws, #Optional
    #project_id=project.id #Optional
)

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero

You can also preview the classification report locally:

In [88]:
my_eval

# Exp 2. Try another prompt

Let's try writing a more detailed prompt:

In [89]:
# 1. Name the experiment <- new name
name = "detailed_prompt"

# 2. Define LLM judge prompt template  <- new prompt
feedback_quality_2 = BinaryClassificationPromptTemplate(
    pre_messages=[("system", "You are evaluating the quality of code reviews given to junior developers.")],
    criteria="""
    A review is **GOOD** if it is actionable and constructive. It should:
    - Offer clear, specific suggestions or highlight issues in a way that the developer can address
    - Be respectful and encourage learning or improvement
    - Use professional, helpful language—even when pointing out problems

    A review is **BAD** if it is non-actionable or overly critical. For example:
    - It may be vague, generic, or hedged to the point of being unhelpful
    - It may focus on praise only, without offering guidance
    - It may sound dismissive, contradictory, harsh, or robotic
    - It may raise a concern but fail to explain what should be done
    """,
    target_category="bad",
    non_target_category="good",
    uncertainty="unknown",
    include_reasoning=True,
)

# 3. Apply the LLM judge
eval_dataset = Dataset.from_pandas(
    pd.DataFrame(review_dataset),
    data_definition=definition,
    descriptors=[
        LLMEval("Generated review",
                template=feedback_quality_2,
                provider="openai",  # Use OpenAI provider with Ollama's OpenAI-compatible API
                model=OLLAMA_LLM_MODEL,
                alias="LLM-judged quality")
    ],
    options=OLLAMA_OPTIONS
)

# 4. Add TRUE/FALSE for judge alignment
eval_dataset.add_descriptors([
    ExactMatch(columns=["LLM-judged quality", "Expert label"], alias="Judge_alignment")
])

[No output generated]

Evaluate the LLM judge quality:

In [90]:
my_eval = run_classification_report(
    eval_dataset,
    name=name,
   #cloud_ws=ws, #Optional
    #project_id=project.id #Optional
)

[No output generated]

In [91]:
my_eval

# Exp 3. Can we make it better?

In [92]:
# 1. Name the experiment <- new name
name = "detailed_prompt_think_better"

# 2. Define LLM judge prompt template  <- new prompt
feedback_quality_3 = BinaryClassificationPromptTemplate(
    pre_messages=[
        ("system", "You are evaluating the quality of code reviews given to junior developers.")],
    criteria="""
    A review is **GOOD** if it is actionable and constructive. It should:
    - Offer clear, specific suggestions or highlight issues in a way that the developer can address
    - Be respectful and encourage learning or improvement
    - Use professional, helpful language—even when pointing out problems

    A review is **BAD** if it is non-actionable or overly critical. For example:
    - It may be vague, generic, or hedged to the point of being unhelpful
    - It may focus on praise only, without offering guidance
    - It may sound dismissive, contradictory, harsh, or robotic
    - It may raise a concern but fail to explain what should be done

    Always explain your reasoning.
    """,
    target_category="bad",
    non_target_category="good",
    uncertainty="unknown",
    include_reasoning=True,
)

# 3. Apply the LLM judge
eval_dataset = Dataset.from_pandas(
    pd.DataFrame(review_dataset),
    data_definition=definition,
    descriptors=[
        LLMEval("Generated review",
                template=feedback_quality_3,
                provider="openai",  # Use OpenAI provider with Ollama's OpenAI-compatible API
                model=OLLAMA_LLM_MODEL,
                alias="LLM-judged quality")
    ],
    options=OLLAMA_OPTIONS
)

# 4. Add TRUE/FALSE for judge alignment
eval_dataset.add_descriptors([
    ExactMatch(columns=["LLM-judged quality", "Expert label"], alias="Judge_alignment")
])

[No output generated]

In [93]:
my_eval = run_classification_report(
    eval_dataset,
    name=name,
    #cloud_ws=ws, #Optional
    #project_id=project.id #Optional
)

[No output generated]

In [94]:
my_eval

# Exp 4. Try a different model (Turbo)

Can a cheaper, simpler model perform as well?

In [95]:
# 1. Name the experiment  <- new name
name = "ollama_local"

# 2. Define LLM judge prompt template
feedback_quality_3 = BinaryClassificationPromptTemplate(
    pre_messages=[
        ("system", "You are evaluating the quality of code reviews given to junior developers.")],
    criteria="""
    A review is **GOOD** if it is actionable and constructive. It should:
    - Offer clear, specific suggestions or highlight issues in a way that the developer can address
    - Be respectful and encourage learning or improvement
    - Use professional, helpful language—even when pointing out problems

    A review is **BAD** if it is non-actionable or overly critical. For example:
    - It may be vague, generic, or hedged to the point of being unhelpful
    - It may focus on praise only, without offering guidance
    - It may sound dismissive, contradictory, harsh, or robotic
    - It may raise a concern but fail to explain what should be done

    Always explain your reasoning.
    """,
    target_category="bad",
    non_target_category="good",
    uncertainty="unknown",
    include_reasoning=True,
)

# 3. Apply the LLM judge
eval_dataset = Dataset.from_pandas(
    pd.DataFrame(review_dataset),
    data_definition=definition,
    descriptors=[
        LLMEval("Generated review",
                template=feedback_quality_3,
                provider="openai",  # Use OpenAI provider with Ollama's OpenAI-compatible API
                model=OLLAMA_LLM_MODEL,
                alias="LLM-judged quality")
    ],
    options=OLLAMA_OPTIONS
)

# 4. Add TRUE/FALSE for judge alignment
eval_dataset.add_descriptors([
    ExactMatch(columns=["LLM-judged quality", "Expert label"], alias="Judge_alignment")
])

[No output generated]

In [96]:
my_eval = run_classification_report(
    eval_dataset,
    name=name,
    #cloud_ws=ws, #Optional
    #project_id=project.id #Optional
)

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero

In [97]:
my_eval

In [98]:
# 1. Name the experiment <- new name
name = "ollama_final"

# 2. Define LLM judge prompt template
feedback_quality_3 = BinaryClassificationPromptTemplate(
    pre_messages=[
        ("system", "You are evaluating the quality of code reviews given to junior developers.")],
    criteria="""
    A review is **GOOD** if it is actionable and constructive. It should:
    - Offer clear, specific suggestions or highlight issues in a way that the developer can address
    - Be respectful and encourage learning or improvement
    - Use professional, helpful language—even when pointing out problems

    A review is **BAD** if it is non-actionable or overly critical. For example:
    - It may be vague, generic, or hedged to the point of being unhelpful
    - It may focus on praise only, without offering guidance
    - It may sound dismissive, contradictory, harsh, or robotic
    - It may raise a concern but fail to explain what should be done

    Always explain your reasoning.
    """,
    target_category="bad",
    non_target_category="good",
    uncertainty="unknown",
    include_reasoning=True,
)

# 3. Apply the LLM judge
eval_dataset = Dataset.from_pandas(
    pd.DataFrame(review_dataset),
    data_definition=definition,
    descriptors=[
        LLMEval("Generated review",
                template=feedback_quality_3,
                provider="openai",  # Use OpenAI provider with Ollama's OpenAI-compatible API
                model=OLLAMA_LLM_MODEL,
                alias="LLM-judged quality")
    ],
    options=OLLAMA_OPTIONS
)

# 4. Add TRUE/FALSE for judge alignment
eval_dataset.add_descriptors([
    ExactMatch(columns=["LLM-judged quality", "Expert label"], alias="Judge_alignment")
])

[No output generated]

In [99]:
my_eval = run_classification_report(
    eval_dataset,
    name=name,
    #cloud_ws=ws, #Optional
    #project_id=project.id #Optional
)

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero

In [ ]:
# === Unload Ollama Model & Shutdown Kernel ===
# Unloads the model from GPU memory before shutting down

try:
    import ollama
    import importlib; importlib.reload(ollama)
    print(f"Unloading Ollama model: {OLLAMA_LLM_MODEL}")
    ollama.generate(model=OLLAMA_LLM_MODEL, prompt="", keep_alive=0)
    print("Model unloaded from GPU memory")
except Exception as e:
    print(f"Model unload skipped: {e}")

# Shut down the kernel to fully release resources
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(restart=False)